# Preprocessing pipeline visualization + audio playback

Παρακολουθεί τι κάνει ΑΚΡΙΒΩΣ το preprocessing σε κάθε βήμα. Δείχνει waveforms + spectrograms + **playable audio** σε κάθε στάδιο ώστε να επαληθεύσουμε ότι δεν καταστρέφει το σήμα.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import librosa
import librosa.display
import soundfile as sf
import noisereduce as nr
import matplotlib.pyplot as plt
from pathlib import Path
import parselmouth
from parselmouth.praat import call
from IPython.display import Audio, display

TARGET_SR = 44100
TRIM_TOP_DB = 25
TARGET_RMS_DB = -25

In [ ]:
def compute_hnr(y, sr=44100):
    """HNR από parselmouth."""
    tmp = '/tmp/_hnr_temp.wav'
    sf.write(tmp, y, sr)
    sound = parselmouth.Sound(tmp)
    try:
        h = sound.to_harmonicity_ac()
        return float(call(h, 'Get mean', 0, 0))
    except Exception:
        return -10

def show_step(y, sr, title):
    """Plot waveform + spectrogram + audio player για ένα στάδιο."""
    rms_db = 20 * np.log10(np.sqrt(np.mean(y**2)) + 1e-10)
    hnr = compute_hnr(y, sr)
    duration = len(y) / sr
    
    print(f'\n=== {title} ===')
    print(f'Duration: {duration:.2f}s · RMS: {rms_db:.1f} dB · HNR: {hnr:.2f} dB · samples: {len(y)}')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3))
    t = np.arange(len(y)) / sr
    ax1.plot(t, y, linewidth=0.5)
    ax1.set_xlim(0, duration)
    ax1.set_ylim(-1.1, 1.1)
    ax1.set_xlabel('Time (s)')
    ax1.set_title(f'{title} — waveform')
    ax1.grid(alpha=0.3)
    
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=2048)), ref=np.max)
    librosa.display.specshow(S, sr=sr, x_axis='time', y_axis='log', ax=ax2, cmap='magma')
    ax2.set_title(f'{title} — spectrogram')
    ax2.set_ylim(80, 8000)
    
    plt.tight_layout()
    plt.show()
    
    display(Audio(y, rate=sr))

## Διάλεξε WAV file

Μπορείς να αλλάξεις το `wav_path`:
- Iyer HC: `../data/iyer/HC_AH/<filename>.wav`
- Iyer PD: `../data/iyer/PD_AH/<filename>.wav`
- Δικό σου: κατέβασε το recording από το browser (download button στο app) και βάλε το path

In [ ]:
# Default: ένα Iyer HC sample
wav_path = Path('../data/iyer/HC_AH/AH_064F_7AB034C9-72E4-438B-A9B3-AD7FDA1596C5.wav')
# Άλλες επιλογές:
# wav_path = next(Path('../data/iyer/PD_AH').glob('*.wav'))  # ένα random PD
# wav_path = Path('/Users/georgegkikas/Downloads/recording_step1_xxx.wav')  # δικό σου

print(f'File: {wav_path.name}')
print(f'Exists: {wav_path.exists()}')
if wav_path.exists():
    print(f'Size: {wav_path.stat().st_size / 1024:.1f} KB')

## Step 0: Original audio

In [ ]:
y_orig, sr = librosa.load(str(wav_path), sr=TARGET_SR, mono=True)
show_step(y_orig, sr, '0. Original audio')

## Step 1: Noise reduction

Ανιχνεύουμε αυτόματα σιωπηλά frames για noise sample.

In [ ]:
intervals = librosa.effects.split(y_orig, top_db=20, frame_length=2048, hop_length=512)
silent_mask = np.ones(len(y_orig), dtype=bool)
for start, end in intervals:
    silent_mask[start:end] = False
noise_sample = y_orig[silent_mask]
print(f'Auto-detected silence: {len(noise_sample)/sr:.2f}s ({len(noise_sample)/len(y_orig)*100:.1f}% of audio)')

if len(noise_sample) > sr * 0.2:
    y_denoised = nr.reduce_noise(y=y_orig, sr=sr, y_noise=noise_sample, stationary=True, prop_decrease=0.7)
    method = 'auto-detected silence as noise reference'
else:
    y_denoised = nr.reduce_noise(y=y_orig, sr=sr, stationary=False, prop_decrease=0.5)
    method = 'non-stationary mode (no silence found)'
print(f'Method: {method}')

show_step(y_denoised, sr, '1. After noise reduction')

## Step 2: Silence trim

In [ ]:
y_trimmed, _ = librosa.effects.trim(y_denoised, top_db=TRIM_TOP_DB)
print(f'Trimmed {(len(y_denoised) - len(y_trimmed))/sr:.2f}s of silence')
show_step(y_trimmed, sr, '2. After silence trim')

## Step 3: RMS normalize

In [ ]:
current_rms = np.sqrt(np.mean(y_trimmed ** 2))
if current_rms > 1e-6:
    target_rms_linear = 10 ** (TARGET_RMS_DB / 20)
    gain = target_rms_linear / current_rms
    y_normalized = y_trimmed * gain
    print(f'Applied gain: {20*np.log10(gain):.2f} dB')
else:
    y_normalized = y_trimmed
    print('Signal too quiet, no normalization')

show_step(y_normalized, sr, '3. After RMS normalize')

## Step 4: Peak clip σε [-1, 1]

In [ ]:
peak = np.max(np.abs(y_normalized))
print(f'Peak before: {peak:.3f}')
if peak > 1.0:
    y_final = y_normalized / peak
    print(f'Clipped to [-1, 1]')
else:
    y_final = y_normalized
    print('No clipping needed')

show_step(y_final, sr, '4. Final processed audio')

## Side-by-side σύγκριση: original vs final

In [ ]:
print('Original:')
display(Audio(y_orig, rate=sr))
print('Final (processed):')
display(Audio(y_final, rate=sr))

## Feature extraction comparison

Δες πώς αλλάζουν τα features πριν και μετά το preprocessing.

In [ ]:
from src.features import extract_features

# Save raw + processed σε temp WAV files
raw_tmp = '/tmp/raw_for_features.wav'
proc_tmp = '/tmp/proc_for_features.wav'
sf.write(raw_tmp, y_orig, sr)
sf.write(proc_tmp, y_final, sr)

feats_raw = extract_features(raw_tmp)
feats_proc = extract_features(proc_tmp)

key = ['locPctJitter', 'locShimmer', 'meanHarmToNoiseHarmonicity',
       'meanIntensity', 'f1', 'b1',
       'mean_MFCC_0th_coef', 'mean_MFCC_2nd_coef', 'std_MFCC_2nd_coef']

print(f'{"Feature":<32s} {"Raw":<12s} {"Processed":<12s} {"Change":<10s}')
print('-' * 70)
for f in key:
    r, p = feats_raw[f], feats_proc[f]
    change = '↑' if p > r else ('↓' if p < r else '=')
    print(f'{f:<32s} {r:<12.4f} {p:<12.4f} {change}')